#  AstroLoc-ML · Colab Training (end-to-end, no Drive)

Idempotent — safe to re-run any cell or **Runtime → Run all**.

**Setup checklist:**
1. `Runtime → Change runtime type → T4 GPU` (free tier) or **L4 / A100** (Pro).
2. Edit `REPO` in cell 2 to your GitHub repo (just `user/repo`, no URL).
3. Click `Runtime → Run all`. ~20 min on T4.

**⚠️ No Drive caching:** if Colab disconnects, you lose everything in `/content/`.
Re-running the notebook re-downloads HYG (~5 s) and re-trains from scratch.
Always run the final cell to download `best.pt` to your laptop after training.

## 1 · GPU sanity check

In [1]:
!nvidia-smi 2>/dev/null | head -16 || echo '❌ No GPU. Runtime > Change runtime type > T4 GPU.'
import torch
print(f'\ntorch={torch.__version__}  cuda_available={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu={torch.cuda.get_device_name(0)}')

Tue May 26 21:20:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2 · Clone (or update) the repo

`REPO` must be **just `user/repo`** — no `https://github.com/`, no `.git`.
Example: `bachnguyennn/astro-ml`.

In [2]:
REPO = 'bachnguyennn/astro-ml'   # <-- EDIT ME (just user/repo)

import pathlib
%cd /content
repo_dir = pathlib.Path('astroloc-ml')
if repo_dir.exists():
    print('Repo already cloned — pulling latest...')
    !cd astroloc-ml && git pull --ff-only
else:
    !git clone https://github.com/{REPO}.git astroloc-ml
%cd astroloc-ml
!git log --oneline -3
!ls

/content
Repo already cloned — pulling latest...
Already up to date.
/content/astroloc-ml
314977f (HEAD -> main, origin/main, origin/HEAD) Rewrite Colab notebook: idempotent, Drive-cached, resilient to restarts
cc12890 Add CLI overrides for fast Colab training
10ecfea Initial commit: AstroLoc-ML
checkpoints  data	  LICENSE    README.md	requirements.txt  src
configs      evaluate.py  notebooks  reports	scripts		  train.py


## 3 · Install Python deps

Colab already has torch + torchvision + numpy + pandas + matplotlib. We only need extras.

In [3]:
!pip install --quiet astropy seaborn python-dotenv pyyaml piexif

## 4 · Download HYG star catalog (~32 MB)

In [4]:
import pathlib
local_path = pathlib.Path('data/catalogs/hygdata_v3.csv')
local_path.parent.mkdir(parents=True, exist_ok=True)

if local_path.exists() and local_path.stat().st_size > 1_000_000:
    print(f'✅ Already on disk: {local_path.stat().st_size/1e6:.1f} MB')
else:
    print('Downloading HYG v41 (~32 MB)...')
    !curl -sL -o {local_path} https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv
    print(f'✅ Downloaded: {local_path.stat().st_size/1e6:.1f} MB')

!head -1 {local_path} | cut -c1-120; echo

✅ Downloaded: 33.9 MB
"id","hip","hd","hr","gl","bf","proper","ra","dec","dist","pmra","pmdec","rv","mag","absmag","spect","ci","x","y","z","v



## 5 · Pipeline sanity check

Loads the catalog, renders one sample, and runs a forward pass.
If anything fails here, fix it before launching training.

In [5]:
import sys, time, torch, numpy as np
sys.path.insert(0, '/content/astroloc-ml')
from src.data.catalog import load_hyg_catalog
from src.data.renderer import render_star_field
from src.models.astrolocnet import AstroLocNet

cat = load_hyg_catalog('data/catalogs/hygdata_v3.csv', mag_limit=8.0)
print(f'Loaded {len(cat):,} stars')

t = time.perf_counter()
img = render_star_field(85.0, -2.0, 30.0, 0.0, cat, image_size=224, rng=np.random.default_rng(0))
print(f'Rendered one star field in {(time.perf_counter()-t)*1000:.0f} ms, shape={img.shape}')

model = AstroLocNet(pretrained=False).cuda()
x = torch.randn(2, 3, 224, 224, device='cuda')
y = model(x)
print(f'Forward pass OK: in={tuple(x.shape)} out={tuple(y.shape)} (B, [ra, dec, rot, log_scale])')

Loaded 41,487 stars
Rendered one star field in 72 ms, shape=(224, 224, 3)
Forward pass OK: in=(2, 3, 224, 224) out=(2, 4) (B, [ra, dec, rot, log_scale])


## 6 · Confirm the new CLI flags are present

If you don't see `--train-samples` and `--num-workers` below, your repo is missing the
latest commit — push from your laptop and re-run cell 2.

In [6]:
!python train.py --help 2>&1 | grep -E 'train-samples|num-workers|epochs-phase' || \
  echo '❌ Missing flags. Run `git push` on your laptop, then re-run cell 2.'

                [--device DEVICE] [--train-samples TRAIN_SAMPLES]
                [--val-samples VAL_SAMPLES] [--epochs-phase1 EPOCHS_PHASE1]
                [--epochs-phase2 EPOCHS_PHASE2] [--num-workers NUM_WORKERS]
  --train-samples TRAIN_SAMPLES
  --epochs-phase1 EPOCHS_PHASE1
  --epochs-phase2 EPOCHS_PHASE2
  --num-workers NUM_WORKERS


## 7 · Smoke training (~30 s on T4)

Verifies the full train loop runs end-to-end with tiny data. Skip if you've done it before.

In [7]:
!python train.py --config configs/default.yaml --smoke --skip-phase3 --device cuda

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 110MB/s] 
[train.py] device=cuda amp=True train_n=256 val_n=64
[train.py] === Phase 1 (head only, backbone frozen) ===
{"phase": "phase1", "epoch": 1, "lr": [0.0], "train": {"ang_sep_mean_deg": 87.29188537597656, "ang_sep_median_deg": 87.41678619384766, "pct_within_5_deg": 0.390625, "pct_within_1_deg": 0.0, "rotation_mae_deg": 91.82088470458984, "scale_mae_deg": Infinity, "loss": 2.17791049182415}, "val": {"ang_sep_mean_deg": 87.52061462402344, "ang_sep_median_deg": 98.72015380859375, "pct_within_5_deg": 0.0, "pct_within_1_deg": 0.0, "rotation_mae_deg": 85.40617370605469, "scale_mae_deg": 45.42729949951172, "loss": 2.7396199703216553}}
[train.py] === Phase 2 (unfreeze last 3 blocks) ===
{"phase": "phase2", "epoch": 1, "lr": [0.0, 0.0], "train": {"ang_sep_mean_deg": 87.24962615966797, "ang_se

## 8 · Real training run

Training is **CPU-bound** (synthetic renderer runs in DataLoader workers on Colab's ~2 vCPUs),
not GPU-bound. Wall-clock scales linearly with `--train-samples`.

| Preset            | `--train-samples` | Epochs (p1/p2) | T4 wall-clock |
| ----------------- | ----------------- | -------------- | ------------- |
| **Fast (active)** | 5,000             | 3 / 6          | **~15–25 min**|
| Standard          | 20,000            | 5 / 10         | ~1.5–2 h      |
| Full (default)    | 50,000            | 5 / 15         | ~5–7 h        |

Output is streamed live — you'll see JSON lines like `{"phase":"phase1","epoch":1,...}` every
couple minutes. The `-u` (unbuffered) flag is important; **never pipe through `tail`** —
it buffers stdout and hides progress.

In [ ]:
# === FAST preset (recommended for first run) — ~15-25 min on T4 ===
!python -u train.py --config configs/default.yaml --device cuda --skip-phase3 \
  --train-samples 5000 --val-samples 500 --num-workers 2 \
  --epochs-phase1 3 --epochs-phase2 6

# === STANDARD preset (~1.5-2 h) — uncomment to use instead ===
# !python -u train.py --config configs/default.yaml --device cuda --skip-phase3 \
#   --train-samples 20000 --val-samples 1000 --num-workers 2 \
#   --epochs-phase1 5 --epochs-phase2 10

# === FULL preset (~5-7 h) — only if you really want to push it ===
# !python -u train.py --config configs/default.yaml --device cuda --skip-phase3 --num-workers 2

[train.py] device=cuda amp=True train_n=5000 val_n=500
[train.py] === Phase 1 (head only, backbone frozen) ===
{"phase": "phase1", "epoch": 1, "lr": [0.00075], "train": {"ang_sep_mean_deg": 89.72183990478516, "ang_sep_median_deg": 89.19805908203125, "pct_within_5_deg": 0.14022436225786805, "pct_within_1_deg": 0.020032051543239504, "rotation_mae_deg": 89.32018280029297, "scale_mae_deg": 54.546913146972656, "loss": 1.8131500665958111}, "val": {"ang_sep_mean_deg": 85.587890625, "ang_sep_median_deg": 85.44906616210938, "pct_within_5_deg": 0.4000000189989805, "pct_within_1_deg": 0.0, "rotation_mae_deg": 89.5569839477539, "scale_mae_deg": 32.108367919921875, "loss": 1.7947230860590935}}
{"phase": "phase1", "epoch": 2, "lr": [0.0002500000000000001], "train": {"ang_sep_mean_deg": 88.32865142822266, "ang_sep_median_deg": 86.23155975341797, "pct_within_5_deg": 0.10016026208177209, "pct_within_1_deg": 0.0, "rotation_mae_deg": 89.40863037109375, "scale_mae_deg": 33.00008010864258, "loss": 1.712482

## 9 · Inspect the trained checkpoint

In [ ]:
import torch, json
ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
print('Best val angular separation (deg):', round(ckpt['best_metric'], 3))
print('History epochs:', len(ckpt.get('history', [])))
if ckpt.get('history'):
    last = ckpt['history'][-1]
    print('Last entry:', json.dumps({'phase': last['phase'], 'epoch': last['epoch'],
                                     'val': last['val']}, indent=2, default=str))

## 10 · Regenerate README figures from the trained model

In [ ]:
!python scripts/generate_readme_figures.py
!ls -lh reports/figures/

In [ ]:
from IPython.display import Image, display
for p in ['reports/figures/05_training_curves.png',
          'reports/figures/09_demo_overlay_0.png',
          'reports/figures/09_demo_overlay_1.png',
          'reports/figures/09_demo_overlay_2.png']:
    display(Image(p))

## 11 · ⚠️ Download checkpoint to your laptop NOW

Without Drive caching, this is your **only** copy of the trained weights. Run this immediately
after training finishes — Colab can disconnect at any time.

Saves `best.pt` to your browser's Downloads folder (~20 MB).

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')

## 12 · Optional: download figures bundle as a zip

In [ ]:
import shutil
shutil.make_archive('astroloc_figures', 'zip', 'reports/figures')
from google.colab import files
files.download('astroloc_figures.zip')

---
## Reconnecting after a disconnect

If Colab killed your session everything in `/content/` is gone. Just **Runtime → Run all**
again — it'll re-clone, re-download HYG, and re-train. No state to recover (this is the
tradeoff for skipping Drive).

If you want the resilient version (HYG cached, checkpoint auto-saved), switch back to the
Drive-enabled version of this notebook from git history.